In [21]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.sqlite import SqliteSaver

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect(
    "memory.db", 
    check_same_thread=False
)

config = {
    "configurable": {
        "thread_id": "1"
    }
}


In [22]:


class State(MessagesState):
    pass

graph_builder = StateGraph(State)


In [23]:

def chatbot(state: State):
    response = llm.invoke(state['messages'])
    return {
        "messages": [response]
    }

In [24]:
graph_builder.add_node("chatbot", chatbot)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)


In [25]:
result  = graph.invoke(
    {
        'messages': [
        {
            "role": "user",
            "content": "I live in Korea now. And the city I live in is Seoul"
        },
        ]
    },
    config=config
)

In [26]:
for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

안녕 내 이름은 진모리야
================================== Ai Message ==================================

안녕하세요, 진모리님! 반갑습니다. 어떻게 도와드릴까요?
================================ Human Message =================================

I live in Korea now. And the city I live in is Seoul
================================== Ai Message ==================================

That's great! Seoul is a vibrant city with a rich history and a lively culture. What's your favorite thing about living in Seoul? Or is there something specific you want to know or discuss about the city?
================================ Human Message =================================

I live in Korea now. And the city I live in is Seoul
================================== Ai Message ==================================

That’s wonderful! Seoul is a dynamic city with so much to offer, from its rich history and culture to its modern amenities and technology. Do you have 

In [27]:
state_history = graph.get_state_history(config)

In [28]:
for state_sanpshot in list(state_history) :
    print(state_sanpshot.next)
    print(state_sanpshot.values["messages"])
    print('---------------------\n')

()
[HumanMessage(content='안녕 내 이름은 진모리야', additional_kwargs={}, response_metadata={}, id='634bc3f7-f075-4e7e-b761-43bf277ff93c'), AIMessage(content='안녕하세요, 진모리님! 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 16, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DLrXv7NxXmjmCERsAa9viM9WxSsLt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d10c8-d036-7c62-8b3d-a6e0dc1309d2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 19, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

In [31]:
state_history = graph.get_state_history(config)

to_fork = list(state_history)[-5]

to_fork.values['messages']
to_fork.config


{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12531d-39f9-6026-8003-9e7305ee7225'}}

In [32]:
from langchain_core.messages import HumanMessage

graph.update_state(
    to_fork.config,
    {
        "messages": [
            HumanMessage(
                content="나는 한국에서 살고 도시는 인천에서 살고 있어.",
                id="8dc17be5-150a-442c-be9e-a2dbc454460e"   
            )
        ]
    }
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12533f-02ea-69b1-8004-b05ee38fd992'}}

In [33]:
forked_state = graph.get_state_history(
    {
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f12533f-02ea-69b1-8004-b05ee38fd992'
        }
    }
)

list(forked_state)

[StateSnapshot(values={'messages': [HumanMessage(content='안녕 내 이름은 진모리야', additional_kwargs={}, response_metadata={}, id='634bc3f7-f075-4e7e-b761-43bf277ff93c'), AIMessage(content='안녕하세요, 진모리님! 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 16, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_668125ce6a', 'id': 'chatcmpl-DLrXv7NxXmjmCERsAa9viM9WxSsLt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d10c8-d036-7c62-8b3d-a6e0dc1309d2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 19, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read

In [34]:

result = graph.invoke(None, {
        'configurable': {
            'thread_id': '1',
            'checkpoint_ns': '',
            'checkpoint_id': '1f12533f-02ea-69b1-8004-b05ee38fd992'
        }
    })
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

안녕 내 이름은 진모리야
================================== Ai Message ==================================

안녕하세요, 진모리님! 반갑습니다. 어떻게 도와드릴까요?
================================ Human Message =================================

나는 한국에서 살고 도시는 인천에서 살고 있어.
================================== Ai Message ==================================

인천에 사신다고요? 인천은 바다와 가까워서 경치가 아름답고, 여러 재미있는 장소가 많죠. 인천의 어떤 부분이 가장 좋으신가요?
